In [ ]:
# Cell 1: Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cell 2: Install required packages
!pip install -q gradio transformers peft detoxify matplotlib seaborn plotly pandas accelerate

In [ ]:
# Cell 3: Import libraries
import torch
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from detoxify import Detoxify
import numpy as np
from io import BytesIO
import base64

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
# Cell 4: Define paths and load data
# Model paths
BASE_MODEL = "gpt2"
FINETUNED_MODEL_PATH = "/content/drive/MyDrive/Capstone_Week5_Models/lora_gpt2_week5_final"

# CSV paths
SUMMARY_CSV = "/content/drive/MyDrive/Capstone_Demo_Data/Week6_Summary_Statistics.csv"
DETAILED_CSV = "/content/drive/MyDrive/Capstone_Demo_Data/Week6_Detailed_Results.csv"

# Load CSV files
print("Loading Week 6 results...")
summary_df = pd.read_csv(SUMMARY_CSV)
detailed_df = pd.read_csv(DETAILED_CSV)

print("Summary statistics loaded!")
print(f"Shape: {summary_df.shape}")
print("\nDetailed results loaded!")
print(f"Shape: {detailed_df.shape}")

Loading Week 6 results...
Summary statistics loaded!
Shape: (6, 15)

Detailed results loaded!
Shape: (360, 16)


In [ ]:
# Cell 5: Load base model and fine-tuned LoRA model
print("Loading models...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# Load base GPT-2 model
print("Loading base GPT-2...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load the fine-tuned model with LoRA adapters
print("Loading the fine-tuned model...")
finetuned_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)
finetuned_model = PeftModel.from_pretrained(finetuned_model, FINETUNED_MODEL_PATH)

# Load toxicity detector
print("Loading toxicity detector...")
toxicity_detector = Detoxify('original')

print("\nBase GPT-2 model loaded!")
print("Your fine-tuned model loaded!")
print("Toxicity detector ready!")
print("\nAll models ready to use!")

Loading models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


Base GPT-2 model loaded!
Your fine-tuned model loaded!
Toxicity detector ready!

All models ready to use!


In [ ]:
# Cell 6: Text generation functions
def generate_text(model, prompt, temperature=1.0, top_p=0.9, top_k=50, max_length=100, strategy="nucleus"):
    """Generate text with specified sampling strategy"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Set sampling parameters based on strategy
    if strategy == "greedy":
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    elif strategy == "nucleus":
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )
    elif strategy == "topk":
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

def calculate_toxicity(text):
    """Calculate toxicity score for generated text"""
    results = toxicity_detector.predict(text)
    return results['toxicity']

print("Generation functions ready!")

Generation functions ready!


In [ ]:
# Cell 7: Tab 1 - Interactive Text Generation Lab
def generate_comparison(prompt, temperature, top_p, top_k, strategy, max_length):
    """Generate text from both base and fine-tuned models"""

    if not prompt.strip():
        return "Please enter a prompt!", "Please enter a prompt!", "N/A", "N/A"

    # Generate from base model
    base_output = generate_text(
        base_model, prompt, temperature, top_p, top_k, max_length, strategy
    )
    base_toxicity = calculate_toxicity(base_output)

    # Generate from fine-tuned model
    finetuned_output = generate_text(
        finetuned_model, prompt, temperature, top_p, top_k, max_length, strategy
    )
    finetuned_toxicity = calculate_toxicity(finetuned_output)

    # Format toxicity scores
    base_tox_str = f"Toxicity: {base_toxicity:.4f}"
    finetuned_tox_str = f"Toxicity: {finetuned_toxicity:.4f}"

    return base_output, finetuned_output, base_tox_str, finetuned_tox_str

# Create Tab 1 interface
with gr.Blocks() as tab1:
    gr.Markdown("## Interactive Text Generation Lab")
    gr.Markdown("Compare base GPT-2 vs your fine-tuned model side-by-side")

    with gr.Row():
        prompt_input = gr.Textbox(
            label="Enter your prompt",
            placeholder="The future of artificial intelligence is...",
            lines=3
        )

    with gr.Row():
        strategy_dropdown = gr.Dropdown(
            choices=["greedy", "nucleus", "topk"],
            value="nucleus",
            label="Sampling Strategy"
        )
        max_length_slider = gr.Slider(
            minimum=50, maximum=200, value=100, step=10,
            label="Max Length (tokens)"
        )

    with gr.Row():
        temperature_slider = gr.Slider(
            minimum=0.5, maximum=1.5, value=1.0, step=0.1,
            label="Temperature"
        )
        top_p_slider = gr.Slider(
            minimum=0.8, maximum=1.0, value=0.9, step=0.05,
            label="Top-p (nucleus)"
        )
        top_k_slider = gr.Slider(
            minimum=10, maximum=100, value=50, step=10,
            label="Top-k"
        )

    generate_btn = gr.Button("Generate Text", variant="primary")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Base GPT-2")
            base_output = gr.Textbox(label="Generated Text", lines=8)
            base_toxicity = gr.Textbox(label="Safety Metrics")

        with gr.Column():
            gr.Markdown("### Your Fine-tuned Model")
            finetuned_output = gr.Textbox(label="Generated Text", lines=8)
            finetuned_toxicity = gr.Textbox(label="Safety Metrics")

    generate_btn.click(
        fn=generate_comparison,
        inputs=[prompt_input, temperature_slider, top_p_slider, top_k_slider,
                strategy_dropdown, max_length_slider],
        outputs=[base_output, finetuned_output, base_toxicity, finetuned_toxicity]
    )

print("Tab 1 interface created!")

Tab 1 interface created!


In [ ]:
# Cell 8: Tab 2 - Performance Analytics Dashboard
def create_performance_charts():
    """Create visualizations from Week 6 results"""

    # Parse the summary data (skip the header rows)
    configs = ['baseline_greedy', 'finetuned_conservative', 'finetuned_greedy',
               'finetuned_nucleus', 'finetuned_topk']

    # Extract metrics from your summary_df
    metrics_data = {
        'Config': configs,
        'Toxicity': [0.0043, 0.0029, 0.0067, 0.0012, 0.0016],
        'Repetition': [0.1808, 0.4310, 0.2148, 0.0510, 0.0680],
        'Diversity': [0.8192, 0.5690, 0.7852, 0.9490, 0.9360]
    }

    # Create figure with 3 subplots
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Plot 1: Toxicity Comparison
    axes[0].bar(range(len(configs)), metrics_data['Toxicity'], color='steelblue')
    axes[0].set_xlabel('Configuration')
    axes[0].set_ylabel('Toxicity Score')
    axes[0].set_title('Toxicity Comparison')
    axes[0].set_xticks(range(len(configs)))
    axes[0].set_xticklabels(configs, rotation=45, ha='right')
    axes[0].axhline(y=0.01, color='r', linestyle='--', label='Safety threshold')
    axes[0].legend()

    # Plot 2: Repetition Ratio
    axes[1].bar(range(len(configs)), metrics_data['Repetition'], color='coral')
    axes[1].set_xlabel('Configuration')
    axes[1].set_ylabel('Repetition Ratio')
    axes[1].set_title('Repetition Analysis')
    axes[1].set_xticks(range(len(configs)))
    axes[1].set_xticklabels(configs, rotation=45, ha='right')

    # Plot 3: Diversity Score
    axes[2].bar(range(len(configs)), metrics_data['Diversity'], color='seagreen')
    axes[2].set_xlabel('Configuration')
    axes[2].set_ylabel('Diversity (distinct-1)')
    axes[2].set_title('Vocabulary Diversity')
    axes[2].set_xticks(range(len(configs)))
    axes[2].set_xticklabels(configs, rotation=45, ha='right')

    plt.tight_layout()

    return fig

# Create Tab 2 interface
with gr.Blocks() as tab2:
    gr.Markdown("## Performance Analytics Dashboard")
    gr.Markdown("Results from Week 6 A/B Testing and Evaluation")

    gr.Markdown("### Key Findings:")
    gr.Markdown("""
    - **71.1% Toxicity Reduction**: Fine-tuned nucleus sampling achieved lowest toxicity (0.0012)
    - **Best Diversity**: Nucleus sampling (top-p=0.9) reached 94.9% unique words
    - **Repetition Control**: Nucleus sampling reduced repetition to 5.1% vs 18% baseline
    - **Optimal Configuration**: LoRA r=8, alpha=16, nucleus sampling top-p=0.9
    """)

    plot_output = gr.Plot(label="Performance Metrics Comparison")

    generate_plots_btn = gr.Button("Generate Performance Charts", variant="primary")

    generate_plots_btn.click(
        fn=create_performance_charts,
        inputs=[],
        outputs=plot_output
    )

print("Tab 2 interface created!")

Tab 2 interface created!


In [ ]:
# Cell 9: Tab 3 - Attention Visualization
def generate_with_attention_info(prompt, max_length=50):
    """Generate text and provide attention-related information"""

    if not prompt.strip():
        return "Please enter a prompt!", "N/A"

    # Generate text
    generated = generate_text(
        finetuned_model, prompt, temperature=0.9, top_p=0.9,
        max_length=max_length, strategy="nucleus"
    )

    # Get tokens for visualization info
    tokens = tokenizer.encode(generated)
    decoded_tokens = [tokenizer.decode([t]) for t in tokens[:20]]

    # Create simple token display
    token_info = "First 20 tokens generated:\n"
    token_info += " | ".join(decoded_tokens)

    return generated, token_info

# Create Tab 3 interface
with gr.Blocks() as tab3:
    gr.Markdown("## Attention Visualization & Explainability")
    gr.Markdown("Explore how the model processes and generates text")

    gr.Markdown("""
    ### Model Interpretability Features:
    - Token-level generation: See how the model builds text word by word
    - LoRA efficiency: Only 0.24% of parameters fine-tuned (294,912 / 124M)
    - Attention mechanism: 12-layer transformer with multi-head attention
    """)

    with gr.Row():
        attn_prompt = gr.Textbox(
            label="Enter prompt for analysis",
            placeholder="Artificial intelligence will transform...",
            lines=2
        )

    attn_length = gr.Slider(
        minimum=30, maximum=100, value=50, step=10,
        label="Generation Length"
    )

    analyze_btn = gr.Button("Generate & Analyze", variant="primary")

    with gr.Row():
        generated_text = gr.Textbox(label="Generated Text", lines=6)

    with gr.Row():
        token_display = gr.Textbox(label="Token Breakdown", lines=4)

    analyze_btn.click(
        fn=generate_with_attention_info,
        inputs=[attn_prompt, attn_length],
        outputs=[generated_text, token_display]
    )

print("Tab 3 interface created!")

Tab 3 interface created!


In [ ]:
# Cell 10: Tab 4 - Experiment Results Matrix
def create_experiment_table():
    """Create table of LoRA experiments from Week 4"""

    experiments_data = {
        'Experiment': ['Exp 1 (Best)', 'Exp 2', 'Exp 3', 'Exp 4'],
        'LoRA Rank (r)': [8, 4, 16, 8],
        'Learning Rate': ['3e-5', '3e-5', '3e-5', '5e-5'],
        'Trainable Params': ['294,912', '147,456', '589,824', '294,912'],
        'Percent of Total': ['0.24%', '0.12%', '0.47%', '0.24%'],
        'Perplexity': [31.89, 35.53, 33.26, 32.59],
        'Result': ['Optimal', 'Too few params', 'Overfitting', 'LR too high']
    }

    df = pd.DataFrame(experiments_data)
    return df

def create_sampling_comparison():
    """Create sampling strategies comparison table"""

    sampling_data = {
        'Strategy': ['Greedy', 'Conservative (T=0.7)', 'Nucleus (p=0.9)', 'Top-k (k=50)'],
        'Diversity (%)': [78.5, 56.9, 94.9, 93.6],
        'Repetition (%)': [21.5, 43.1, 5.1, 6.8],
        'Toxicity': [0.0067, 0.0029, 0.0012, 0.0016],
        'Best Use Case': ['Factual', 'Safe but limited', 'Creative (BEST)', 'Creative']
    }

    df = pd.DataFrame(sampling_data)
    return df

# Create Tab 4 interface
with gr.Blocks() as tab4:
    gr.Markdown("## Experiment Results & Findings")
    gr.Markdown("Comprehensive overview of all experiments conducted")

    gr.Markdown("### Week 4: LoRA Hyperparameter Optimization")

    lora_table = gr.Dataframe(
        value=create_experiment_table(),
        label="LoRA Configuration Experiments"
    )

    gr.Markdown("""
    **Key Finding**: LoRA rank r=8 with learning rate 3e-5 achieves optimal balance.
    Higher ranks (r=16) led to overfitting despite doubling parameters.
    """)

    gr.Markdown("### Week 6: Sampling Strategy Comparison")

    sampling_table = gr.Dataframe(
        value=create_sampling_comparison(),
        label="Sampling Strategies (360 outputs tested)"
    )

    gr.Markdown("""
    **Key Finding**: Nucleus sampling (top-p=0.9) achieved best overall performance:
    - Highest diversity (94.9% unique words)
    - Lowest repetition (5.1%)
    - Lowest toxicity (0.0012)
    - Statistically significant improvements (t-test p < 0.05)
    """)

    gr.Markdown("### Final Optimal Configuration")
    gr.Markdown("""
    - **Model**: GPT-2 Small (124M parameters)
    - **Fine-tuning**: LoRA r=8, alpha=16 (294,912 trainable params)
    - **Sampling**: Nucleus with top-p=0.9, temperature=0.9
    - **Performance**: 31.89 perplexity, 71% toxicity reduction
    - **Efficiency**: 99.76% parameter savings vs full fine-tuning
    """)

print("Tab 4 interface created!")

Tab 4 interface created!


In [ ]:
# Cell 11: Combine all tabs and launch the dashboard
demo = gr.TabbedInterface(
    [tab1, tab2, tab3, tab4],
    ["Text Generation Lab", "Performance Analytics", "Explainability", "Experiment Results"],
    title="LLM Optimization Research Dashboard",
    theme=gr.themes.Soft()
)

# Launch the dashboard
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://42b63b6815aa35f38e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
